# Violence Against Nursing-Facility Caregivers: What OSHA's 2025 Case Data Shows

Reproduces every figure in [`README.md`](README.md) from OSHA's Injury Tracking Application (ITA) case detail data, and the state and size tables in [`../data/`](../data/).

**Author** Yuxuan Huang · Vantara Medical Equipment (Mahoraga Vantara LLC), New York
**Data year** 2025 (submissions through 2026-08-31)
**Source data** US Government work, 17 U.S.C. §105

---
## 1. Environment

In [1]:
!pip install -q duckdb pandas requests
import os, re, duckdb, requests, pandas as pd, numpy as np
pd.set_option('display.max_columns', None); pd.set_option('display.width', 200)
os.makedirs('data', exist_ok=True); os.makedirs('derived', exist_ok=True)
con = duckdb.connect(':memory:')

---
## 2. Download

OSHA publishes ITA files at [osha.gov/itadata](https://www.osha.gov/itadata). The file name changes with each release, and OSHA removes the earlier file when it republishes; update the URL below from that page. OSHA's server returns HTTP 403 without a browser User-Agent header.

This version uses the release covering submissions through 2026-08-31. Version 1.x of this repository used the earlier release through 2026-03-15, which OSHA has since replaced; counts differ between the two because the later release includes late submissions.

In [2]:
CASES_URL = 'https://www.osha.gov/sites/default/largefiles/ITA_Case_Detail_Data_2025_through_08-31-2026.csv'
HDRS = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 '
                      '(KHTML, like Gecko) Chrome/120.0 Safari/537.36'}
PATH = 'data/osha_cases.csv'
if not os.path.exists(PATH):
    with requests.get(CASES_URL, headers=HDRS, stream=True, timeout=1800) as r:
        r.raise_for_status()
        with open(PATH, 'wb') as f:
            for chunk in r.iter_content(1 << 20): f.write(chunk)
print('downloaded', round(os.path.getsize(PATH) / 1e6), 'MB')

downloaded 475 MB


---
## 3. Load nursing-facility cases (NAICS 6231)

Read verbatim as text. Column names are resolved case-insensitively because OSHA has renamed fields between releases. Day counts use `-9999` as a sentinel; negative values are set to null.

In [3]:
con.execute(f'''CREATE OR REPLACE TABLE raw_cases AS
    SELECT * FROM read_csv_auto('{PATH}', header=true, all_varchar=true, ignore_errors=true, sample_size=-1)''')
cols = [r[0] for r in con.execute('DESCRIBE raw_cases').fetchall()]
print(con.execute('SELECT COUNT(*) FROM raw_cases').fetchone()[0], 'rows, all industries')

def col(*names):
    low = {c.lower(): c for c in cols}
    for n in names:
        if n.lower() in low: return low[n.lower()]
    raise KeyError(names)
F = dict(naics=col('naics_code'), state=col('state'), est=col('establishment_ID'), size=col('size'),
         job=col('job_description'), soc=col('SOC_Description'),
         dafw=col('dafw_num_away'), djtr=col('djtr_num_tr'),
         before=col('New_nar_before_incident'), what=col('New_nar_what_happened'),
         injury=col('New_nar_injury_illness'), obj=col('New_nar_object_substance'))

df = con.execute(f'''
SELECT "{F['state']}" AS state, "{F['est']}" AS est, "{F['size']}" AS size,
       "{F['job']}" AS job, "{F['soc']}" AS soc,
       CASE WHEN TRY_CAST("{F['dafw']}" AS INT) >= 0 THEN TRY_CAST("{F['dafw']}" AS INT) END AS dafw,
       CASE WHEN TRY_CAST("{F['djtr']}" AS INT) >= 0 THEN TRY_CAST("{F['djtr']}" AS INT) END AS djtr,
       lower(concat_ws(' ', "{F['before']}", "{F['what']}", "{F['injury']}", "{F['obj']}")) AS narr
FROM raw_cases WHERE "{F['naics']}" LIKE '6231%'
''').df()
print(f'nursing-facility cases: {len(df):,}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

782582 rows, all industries
nursing-facility cases: 25,860


---
## 4. Classification rules

**Violence.** A named person (resident, patient, client, visitor, family member) followed within a few words by a hostile act; or the act described passively as done *by* such a person; or combative / aggressive behavior together with a hostile act; or an assault by a co-worker. Contact where the employee struck an object ("hit her head on the cabinet") is excluded unless aggression is also described. Resistance during care (pulling back, pushing back) is not counted as violence on its own.

**Resident handling.** Identical to the frozen rule in [`../nursing-facility-handling/`](../nursing-facility-handling/), so that the two modules' counts are consistent.

In [12]:
PERSON = r"(?:resident|residents|patient|patients|pt|res|client|member|visitor|family member)"
HOSTILE = (r"(?:hit|hits|struck|strikes?|punch\w*|kick\w*|bit|bites?|biting|bitten|scratch\w*|slap\w*|spit|spat|"
           r"head[- ]?butt\w*|pinch\w*|squeez\w*|dug|swung at|wrapped (?:her |his |their )?arms|"
           r"grabb?(?:ed|ing|s)? (?:the )?(?:employee|ee|staff|her|his|my|onto|arms?|forearm|hand|wrist|rt|lt|right|left)\b|"
           r"twist(?:ed|ing) (?:the )?(?:employee|ee|her|his|my)\b(?! (?:right |left )?(?:knee|ankle|back|foot))|push(?:ed|ing) (?:the )?(?:employee|ee|staff|her|him)\b(?! back)|"
           r"spray\w*|assault\w*|attack\w*)")
A = rf"\b{PERSON}\b(?:\W+\w+){{0,5}}?\W+{HOSTILE}"
PASSIVE = rf"\b(?:bit|bitten|hit|struck|kicked|punched|scratched|attacked|assaulted|grabbed)\b(?:\W+\w+){{0,4}}?\W+by (?:a |the )?{PERSON}\b"
AGG = r"\b(?:combative|aggressive|aggression|agitated|assault\w*|attacked|human bite|physical aggression|resident attack)\b"
AGG_ACT = r"\b(?:pull\w*|grab\w*|punch\w*|hit|kick\w*|bit|bite|scratch\w*|twist\w*|push\w*|swung|squeez\w*)\b"
COWORKER = r"\b(?:co-?workers?|employees?|staff members?)\b(?:\W+\w+){0,3}?\W+(?:jumped|attacked|assaulted|punched|sprayed)\b"
ACC = (r"\b(?:hit|struck|bumped|banged)\b (?:her |his |my |their |the )?(?:left |right |l |r )?"
       r"(?:head|hand|knee|arm|elbow|shin|foot|toe|big toe|leg|shoulder|finger)\b(?:\W+\w+){0,3}?\W+(?:on|against|into)\b")

def is_violence(s):
    s = s or ''
    if re.search(COWORKER, s): return True
    if re.search(A, s) or re.search(PASSIVE, s):
        return not (re.search(ACC, s) and not re.search(AGG, s))
    if re.search(AGG, s) and re.search(AGG_ACT, s): return True
    return False

# resident handling: frozen strict rule from nursing-facility-handling
PERSON_H = r"\b(?:resident|residents|patient|patients|pt|res)\b"
MOVEMENT = (r"\b(?:transfer\w*|lift\w*|reposition\w*|boost\w*|turn\w*|roll\w*|"
            r"pull\w* (?:up|over)|pivot\w*|hoyer|mechanical lift|gait belt|sit[- ]to[- ]stand|"
            r"slide sheet|draw sheet|bed to chair|chair to bed|wheelchair to bed|bed to wheelchair|"
            r"toilet\w* transfer|ambulat\w*)\b")
PHRASE = r"patient handling|resident handling|resident care transfer"

n = df.narr.fillna('')
df['violence'] = n.map(is_violence)
df['handling'] = (n.str.contains(PERSON_H, regex=True) & n.str.contains(MOVEMENT, regex=True)) | n.str.contains(PHRASE, regex=True)
df['violence_during_handling'] = df.violence & df.handling

with open('derived/classification-rule.txt', 'w') as f:
    f.write(f'PERSON = {PERSON}\nHOSTILE = {HOSTILE}\nA = {A}\nPASSIVE = {PASSIVE}\n'
            f'AGG = {AGG}\nAGG_ACT = {AGG_ACT}\nCOWORKER = {COWORKER}\nACC = {ACC}\n\n'
            'violence = COWORKER or ((A or PASSIVE) and not (ACC and not AGG)) or (AGG and AGG_ACT)\n')

v = df[df.violence]
print(f'violence: {len(v):,} ({len(v)/len(df):.1%})')
print(f'handling: {int(df.handling.sum()):,} (should equal the nursing-facility-handling module: 5,654)')
print(f'violence during handling: {int(df.violence_during_handling.sum()):,} '
      f'({df.violence_during_handling.sum()/len(v):.1%} of violence)')

violence: 2,291 (8.9%)
handling: 5,654 (should equal the nursing-facility-handling module: 5,654)
violence during handling: 365 (15.9% of violence)


### Spot check

Twenty random violence narratives. Read them. If the rule misclassifies, fix the rule, not the count.

In [5]:
for t in v.narr.sample(20, random_state=1): print(' -', t[:220])

 - giving a shower resident was aggressive during shower - scratched her arm scratches resident
 - trying to get resident back to bed taking off residents shoes resident kicked ee 's left 2nd finger hyper-extended finger being kicked by resident
 - assisting the resident in a transfer resident became agitated and kicked employee in left knee contusion to l knee resident
 - ee was assisting resident and resident grabbed ee hand twisting fingers ee was assisting resident and resident grabbed ee hand twisting fingers strain twisted fingers resident
 - employee was giving a shower to a resident employee was giving a shower to a resident when the resident bit her right middle finger causing a cut. right finger(s): bite na
 - resident care resident hit ms. [redacted] in the face and chest while she was providing care. head contusion na
 - employee was transferring resident from toilet to wheelchair. resident became agitated and bite right hand of employee. bite without puncture to right hand

---
## 5. Validation against the silver-standard benchmark

Uses [`../benchmark/benchmark-500-ai-prelabeled.csv`](../benchmark/); the cell downloads it from this repository if it is not present. The labels are AI-produced and not human-reviewed (a silver standard). The rule was written while reading these cases, so the result is a development score, not an out-of-sample estimate. The score does not depend on the OSHA release used above: it is computed on the 500 stored narratives only.

In [6]:
BF = '../benchmark/benchmark-500-ai-prelabeled.csv'
if not os.path.exists(BF):
    BF = 'benchmark/benchmark-500-ai-prelabeled.csv'
    if not os.path.exists(BF):
        os.makedirs('benchmark', exist_ok=True)
        url = ('https://raw.githubusercontent.com/yuxuan-huang-data/us-care-safety-signals/'
               'main/benchmark/benchmark-500-ai-prelabeled.csv')
        open(BF, 'wb').write(requests.get(url, timeout=60).content)
if os.path.exists(BF):
    b = pd.read_csv(BF)
    pred = b.narr.fillna('').str.lower().map(is_violence)
    truth = b.ai_labels.str.contains('violence')
    tp, fp, fn = int((pred & truth).sum()), int((pred & ~truth).sum()), int((~pred & truth).sum())
    val = pd.Series({'sample': len(b), 'violence (silver label)': int(truth.sum()), 'rule positive': int(pred.sum()),
                     'precision': round(tp/max(tp+fp, 1), 3), 'recall': round(tp/max(tp+fn, 1), 3)})
    val.rename('value').to_csv('derived/validation-result.csv')
    print(val.to_string())
    print('\n-- false positives --'); [print(' -', t[:180]) for t in b[pred & ~truth].narr]
    print('\n-- false negatives --'); [print(' -', t[:180]) for t in b[~pred & truth].narr]
else:
    print('benchmark file not found; skipping')

sample                     500.000
violence (silver label)     56.000
rule positive               51.000
precision                    0.961
recall                       0.875

-- false positives --
 - assisting resident or individual [redacted] stated combative resident was walking down the hall on a brief and she ran to assist him to keep him from falling. attempted to grab the
 - changing or either tansferring a patient ee reports she hit her [redacted] a mounted tv when changing a resident. contusion skull furniturefixtures

-- false negatives --
 - patient care resident resisting care and pulled away causing arm to pop. assisting another residents fall made pain worse. sprain of right elbow. strain of other muscle fascia and 
 - helping a resident to sit down resident pushed employees [redacted] hand into the wheelchair pain in r hand residents hand into wheelchair
 - caring for resident claimant states that she was holding resident when resident pushed back injuring left hand spra

---
## 6. Who is injured

In [7]:
def occ(r):
    s = f"{r.soc or ''} {r.job or ''}".lower()
    if re.search(r'nursing assistant|\bcna\b|\bstna\b|\bcma\b|aide|orderl', s): return 'Nursing assistant / aide'
    if re.search(r'licensed practical|licensed vocational|\blpn\b|\blvn\b', s): return 'LPN / LVN'
    if re.search(r'registered nurse|\brn\b', s): return 'Registered nurse'
    if re.search(r'therap', s): return 'Therapy'
    if re.search(r'housekeep|janitor|laundry|dietary|cook|food', s): return 'Housekeeping / dietary'
    return 'Other / unspecified'
v = v.assign(occ=v.apply(occ, axis=1))
by_occ = v.occ.value_counts().rename('cases').to_frame().assign(share=lambda d: (d.cases/d.cases.sum()).round(3))
by_occ

,cases,share
occ,,
Nursing assistant / aide,1716,0.749
LPN / LVN,210,0.092
Other / unspecified,174,0.076
Registered nurse,143,0.062
Housekeeping / dietary,26,0.011
Therapy,22,0.010


---
## 7. Mechanism and body region

Keyword counts; one case can name more than one.

In [8]:
MECH = {'hit / punch / strike': r'\b(?:hit|struck|strike|punch\w*|slap\w*|swung at)\b', 'kick': r'\bkick\w*',
        'bite': r'\b(?:bit|bite|biting|bitten)\b', 'scratch': r'\bscratch\w*',
        'grab / twist / pinch': r'\b(?:grab\w*|twist\w*|pinch\w*|squeez\w*)', 'push / shove': r'\b(?:push\w*|shov\w*)\b',
        'spit': r'\b(?:spit|spat)\b'}
BODY = {'head / face / eye': r'\b(?:head|face|facial|eye|nose|jaw|mouth|lip|forehead|temple|cheek|ear|concussion)\b',
        'hand / finger / wrist': r'\b(?:hand|finger|thumb|wrist)\b', 'arm / shoulder': r'\b(?:arm|forearm|shoulder|elbow)\b',
        'chest / abdomen': r'\b(?:chest|abdomen|abdominal|stomach|rib|breast)\b', 'back / neck': r'\b(?:back|lumbar|neck|spine)\b',
        'leg / knee / foot': r'\b(?:leg|knee|foot|ankle|shin|thigh)\b'}
tab = lambda D: (pd.Series({k: int(v.narr.str.contains(p, regex=True).sum()) for k, p in D.items()})
                 .rename('cases').to_frame().assign(share=lambda d: (d.cases/len(v)).round(3)))
mech, body = tab(MECH), tab(BODY)
display(mech); display(body)

,cases,share
hit / punch / strike,1009,0.440
kick,278,0.121
bite,336,0.147
scratch,200,0.087
grab / twist / pinch,520,0.227
push / shove,131,0.057
spit,31,0.014


,cases,share
head / face / eye,684,0.299
hand / finger / wrist,635,0.277
arm / shoulder,599,0.261
chest / abdomen,317,0.138
back / neck,429,0.187
leg / knee / foot,247,0.108


---
## 8. Severity

In [9]:
sev = pd.Series({'violence cases': len(v), 'days away (sum)': int(v.dafw.sum()), 'days restricted (sum)': int(v.djtr.sum()),
                 'zero days away': int((v.dafw.fillna(0) == 0).sum()), 'cases > 90 days away': int((v.dafw > 90).sum()),
                 'longest absence': int(v.dafw.max())})
sev

,0
violence cases,2291
days away (sum),17133
days restricted (sum),22596
zero days away,1561
cases > 90 days away,62
longest absence,347


---
## 9. By state and by establishment size

Counts of reported cases. Establishments with no cases are absent from the source file, so these tables do not support rates. Size codes follow the OSHA ITA data dictionary; code 2 was split into 21 and 22 beginning with 2023 data.

In [10]:
st = (df.groupby('state').agg(cases=('narr', 'size'), establishments=('est', 'nunique'),
                              handling=('handling', 'sum'), violence=('violence', 'sum'))
        .assign(handling_share=lambda d: (d.handling/d.cases).round(3), violence_share=lambda d: (d.violence/d.cases).round(3)))
days = df.groupby('state').apply(lambda g: pd.Series({
    'handling_days_away': int(g[g.handling].dafw.sum()), 'handling_days_restricted': int(g[g.handling].djtr.sum()),
    'violence_days_away': int(g[g.violence].dafw.sum()), 'violence_days_restricted': int(g[g.violence].djtr.sum())}),
    include_groups=False)
st = st.join(days).sort_values('cases', ascending=False)
assert st.cases.sum() == len(df) and st.handling.sum() == df.handling.sum() and st.violence.sum() == df.violence.sum()

SIZE = {'1': 'under 20', '2': '20-249 (pre-2023 code)', '21': '20-99', '22': '100-249', '3': '250+'}
df['size_label'] = df['size'].astype(str).str.strip().map(SIZE).fillna('unknown')
sz = (df.groupby('size_label').agg(cases=('narr', 'size'), establishments=('est', 'nunique'),
                                   handling=('handling', 'sum'), violence=('violence', 'sum'))
        .assign(handling_share=lambda d: (d.handling/d.cases).round(3), violence_share=lambda d: (d.violence/d.cases).round(3))
        .reindex(['under 20', '20-99', '20-249 (pre-2023 code)', '100-249', '250+', 'unknown']).dropna(how='all'))
display(st.head(15)); display(sz)

,cases,establishments,handling,violence,handling_share,violence_share,handling_days_away,handling_days_restricted,violence_days_away,violence_days_restricted
state,,,,,,,,,,
NY,2282,226,379,301,0.166,0.132,9049,2185,5666,632
CA,1947,248,473,204,0.243,0.105,7120,11471,687,1060
FL,1817,286,367,200,0.202,0.110,2877,10039,1223,2592
PA,1360,186,299,141,0.220,0.104,2355,7848,735,2577
IL,1321,196,223,123,0.169,0.093,2911,4382,1016,1373
MI,1315,200,313,102,0.238,0.078,2432,7758,549,602
TX,1249,322,280,145,0.224,0.116,1366,5908,366,1650
OH,1098,234,182,77,0.166,0.070,808,3299,172,558
NJ,1015,127,203,105,0.200,0.103,1078,5665,792,1612


,cases,establishments,handling,violence,handling_share,violence_share
size_label,,,,,,
under 20,87.0,11.0,36.0,25.0,0.414,0.287
20-99,781.0,223.0,231.0,49.0,0.296,0.063
20-249 (pre-2023 code),471.0,64.0,81.0,35.0,0.172,0.074
100-249,19748.0,3307.0,4317.0,1721.0,0.219,0.087
250+,4773.0,393.0,989.0,461.0,0.207,0.097


---
## 10. Export

In [11]:
by_occ.to_csv('derived/violence-by-occupation.csv')
sev.rename('value').to_csv('derived/violence-severity.csv')
body.to_csv('derived/violence-body-part.csv'); mech.to_csv('derived/violence-mechanism.csv')
# state and size tables go to ../data/ inside the repository, or to data-tables/ when run on its own (e.g. Colab)
DATA_DIR = '../data' if os.path.isdir('../data') else 'data-tables'
os.makedirs(DATA_DIR, exist_ok=True)
st.to_csv(f'{DATA_DIR}/nf-injury-by-state-2025.csv'); sz.to_csv(f'{DATA_DIR}/nf-injury-by-size-2025.csv')
print('derived:', sorted(os.listdir('derived')))
print(DATA_DIR + ':', sorted(os.listdir(DATA_DIR)))

derived: ['classification-rule.txt', 'validation-result.csv', 'violence-body-part.csv', 'violence-by-occupation.csv', 'violence-mechanism.csv', 'violence-severity.csv']
data-tables: ['nf-injury-by-size-2025.csv', 'nf-injury-by-state-2025.csv']


---
## Limitations

**Coverage.** Case detail submission is required only of establishments with 100 or more employees in designated industries. Small facilities are under-represented.

**Employer-reported.** Narratives are free text written by employers; detail and wording vary.

**Rule-based classification.** Violence is a keyword rule, not an OSHA field. Its development accuracy is reported in Section 5.

**No rates.** Establishments with no cases do not appear in the case file.

**No establishment is named.** OSHA: recording an injury does not mean the employer was at fault or violated any OSHA rule.